# Data Cleaning 

## Skin-Deep Insights: Aspect-Based Sentiment Analysis of Sephora Reviews

**Goal:** Prepare the working dataset for NLP analysis by removing
structurally invalid records, filtering low-quality reviews, and
normalizing text for downstream processing.

**Why NLP cleaning goes beyond standard tabular cleaning:**
Standard data cleaning removes nulls and duplicates. NLP cleaning
additionally requires: removing reviews too short for aspect extraction,
removing reviews with systematic bias (promotional), and normalizing
text so models treat equivalent words identically.

**Two-part approach:**
```
Part 1 — Structural Cleaning
Remove null review_text → Convert dates
→ Inspect duplicates → Remove ingestion duplicates only

Part 2 — Text Quality & Normalization
Filter short reviews (<5 words) → Remove incentivized reviews
→ Lowercase → Remove punctuation → Normalize whitespace
→ Standardize skin_type

**Load Data**

In [1]:
import pandas as pd
df = pd.read_csv("../sephora/working_reviews.csv")

In [2]:
df.shape

(1094411, 8)

In [3]:
df.isnull().sum()

product_id           0
product_name         0
brand_name           0
price_usd            0
rating               0
review_text       1444
skin_type       111557
review_date          0
dtype: int64

**Pre-cleaning null snapshot:**

Only two columns have missing values — `review_text` (1,444) and
`skin_type` (111,557). All other columns are structurally complete.
Our cleaning approach differs by column:
- `review_text` nulls → Remove. No text = no NLP value.
- `skin_type` nulls → Retain. Demographic data should not be imputed.

In [4]:
df = df.dropna(subset=['review_text'])
df.shape

(1092967, 8)

1,444 null review texts removed. These records have no analytical
value — there is no text to extract aspects from. Dropping them
loses 0.13% of data — well within acceptable range.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1092967 entries, 0 to 1094410
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   product_id    1092967 non-null  object 
 1   product_name  1092967 non-null  object 
 2   brand_name    1092967 non-null  object 
 3   price_usd     1092967 non-null  float64
 4   rating        1092967 non-null  int64  
 5   review_text   1092967 non-null  object 
 6   skin_type     981449 non-null   object 
 7   review_date   1092967 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 75.0+ MB


In [6]:
df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1092967 entries, 0 to 1094410
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   product_id    1092967 non-null  object        
 1   product_name  1092967 non-null  object        
 2   brand_name    1092967 non-null  object        
 3   price_usd     1092967 non-null  float64       
 4   rating        1092967 non-null  int64         
 5   review_text   1092967 non-null  object        
 6   skin_type     981449 non-null   object        
 7   review_date   1092967 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 75.0+ MB


#### Duplicate Investigation Strategy

Not all repeated review text is a data error. Two independent users
can write "Love this product" without that being a duplication issue.
We test four levels to identify the correct removal threshold:

- **Level 1:** All 8 columns identical → likely ingestion error
- **Level 2:** Same review_text only → natural language repetition, keep
- **Level 3:** Same product + text → suspicious, investigate
- **Level 4:** Same product + text + rating + date → almost certainly ingestion error, remove

This approach preserves authentic repeated opinions while eliminating
true data ingestion artifacts.

In [8]:
df.duplicated().sum()

np.int64(571)

In [9]:
df.duplicated(subset=['review_text']).sum()

np.int64(123548)

In [10]:
df.duplicated(subset=['product_id', 'review_text']).sum()

np.int64(994)

In [11]:
duplicates_full = df[df.duplicated(keep=False)]
duplicates_full.head(20)

,product_id,product_name,brand_name,price_usd,rating,review_text,skin_type,review_date
877,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,MY FAVORITE LIP SLEEPING MASK. It feels so goo...,oily,2022-08-05
878,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,MY FAVORITE LIP SLEEPING MASK. It feels so goo...,oily,2022-08-05
2136,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,The Aquaphor healing ointment saves my dry cra...,combination,2021-11-07
2137,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,The Aquaphor healing ointment saves my dry cra...,combination,2021-11-07
3854,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,4,Have been using this (the same pot) for the la...,combination,2021-01-10
3855,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,4,Have been using this (the same pot) for the la...,combination,2021-01-10
4166,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,I love this product when I put it on at night....,normal,2020-11-29
4167,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,I love this product when I put it on at night....,normal,2020-11-29
4579,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,Love love love this stuff! It smells so fun an...,combination,2020-09-20
4580,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,Love love love this stuff! It smells so fun an...,combination,2020-09-20


In [12]:
duplicates_full.shape

(1001, 8)

In [13]:
duplicates_product_text = df[df.duplicated(subset=['product_id', 'review_text'], keep=False)]
duplicates_product_text.sort_values(['product_id']).head(20)

,product_id,product_name,brand_name,price_usd,rating,review_text,skin_type,review_date
453690,P12045,Grape Water Moisturizing Face Mist,Caudalie,12.0,4,I’ve only been using this for a few days but i...,dry,2020-12-31
453689,P12045,Grape Water Moisturizing Face Mist,Caudalie,12.0,4,I’ve only been using this for a few days but i...,dry,2020-12-31
1012954,P122651,Clarifying Lotion 1,CLINIQUE,20.0,4,Excellent for dry skin.,dry,2010-06-01
1012955,P122651,Clarifying Lotion 1,CLINIQUE,20.0,4,Excellent for dry skin.,dry,2010-06-01
729763,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05
729764,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05
729765,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05
729766,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05
729767,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05
729768,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,26.0,5,Makes your face soft and it does not dry your ...,NaN,2014-05-05


In [14]:
duplicates_text = df[df.duplicated(subset=['review_text'], keep=False)]
duplicates_text[['product_id', 'review_text']].head(20)

,product_id,review_text
877,P420652,MY FAVORITE LIP SLEEPING MASK. It feels so goo...
878,P420652,MY FAVORITE LIP SLEEPING MASK. It feels so goo...
1586,P420652,"Really nice sunscreen, it hydrated my skin pre..."
2136,P420652,The Aquaphor healing ointment saves my dry cra...
2137,P420652,The Aquaphor healing ointment saves my dry cra...
2498,P420652,Love this sleeping mask!I even had some dark s...
2803,P420652,My favorite Lip Balm ever. My friends and I ha...
3854,P420652,Have been using this (the same pot) for the la...
3855,P420652,Have been using this (the same pot) for the la...
4166,P420652,I love this product when I put it on at night....


In [15]:
df.duplicated(subset=['product_id', 'review_text', 'rating', 'review_date']).sum()

np.int64(650)

**Duplicate Decision:**

| Level | Count | Decision | Reasoning |
|---|---|---|---|
| Full row identical | 571 | Investigate further | May include triplicates |
| Same text only | 123,548 | Keep | Natural language repetition |
| Same product + text | 994 | Investigate | Possible but check dates |
| Same product + text + rating + date | 650 | Remove | Ingestion duplicates |

The 650 records sharing identical product, text, rating AND date represent
near-certain data ingestion errors. The probability of two users writing
identical text about the same product, on the same day, with the same
rating is negligible. All other repeated text is preserved.

In [16]:
df = df.drop_duplicates(subset=['product_id', 'review_text', 'rating', 'review_date'])

In [17]:
df.duplicated(subset=['product_id', 'review_text', 'rating', 'review_date']).sum()

np.int64(0)

650 ingestion duplicates removed. Authentic repeated opinions preserved.
Dataset at 1,092,317 rows.

In [18]:
df.shape

(1092317, 8)

#### Text Quality Filtering

Two categories of low-quality text are removed before NLP:

**1. Short reviews (<5 words):** Reviews like "Love it" or "Bad" contain
sentiment but no aspect information. ABSA requires enough context to
identify what product feature is being discussed. Minimum 5 words is a
conservative threshold that removes only the least informative records.

**2. Incentivized reviews:** Reviews where the author received free
products introduce systematic positive bias. Identifying them by
promotional language patterns removes bias from sentiment analysis.
This is a business decision as much as a technical one.

**Remove short reviews**

In [19]:
df['word_count'] = df['review_text'].str.split().str.len()
df = df[df['word_count'] >= 5]
df.head()

,product_id,product_name,brand_name,price_usd,rating,review_text,skin_type,review_date,word_count
0,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0,5,I use this with the Nudestix “Citrus Clean Bal...,dry,2023-02-01,79
1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,1,I bought this lip mask after reading the revie...,NaN,2023-03-21,28
2,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,My review title says it all! I get so excited ...,dry,2023-03-21,53
3,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,I’ve always loved this formula for a long time...,combination,2023-03-20,45
4,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,5,"If you have dry cracked lips, this is a must h...",combination,2023-03-20,46


In [20]:
df['word_count'].mean()

np.float64(60.62466705237398)

In [21]:
df.shape

(1090637, 9)

In [22]:
df['word_count'].median()

50.0

In [23]:
df['word_count'].min()

5

In [24]:
df['word_count'].max()

1216

In [25]:
df['word_count'].describe()

count    1.090637e+06
mean     6.062467e+01
std      4.345553e+01
min      5.000000e+00
25%      3.300000e+01
50%      5.000000e+01
75%      7.600000e+01
max      1.216000e+03
Name: word_count, dtype: float64

In [26]:
print(df.head())

  product_id                                       product_name brand_name  \
0    P504322                     Gentle Hydra-Gel Face Cleanser   NUDESTIX   
1    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   
2    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   
3    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   
4    P420652  Lip Sleeping Mask Intense Hydration with Vitam...    LANEIGE   

   price_usd  rating                                        review_text  \
0       19.0       5  I use this with the Nudestix “Citrus Clean Bal...   
1       24.0       1  I bought this lip mask after reading the revie...   
2       24.0       5  My review title says it all! I get so excited ...   
3       24.0       5  I’ve always loved this formula for a long time...   
4       24.0       5  If you have dry cracked lips, this is a must h...   

     skin_type review_date  word_count  
0          dry  2023-02-01          79 

In [27]:
print("Rows after filtering:", len(df))

Rows after filtering: 1090637


In [28]:
df['review_text'].str.split().str.len().describe()

count    1.090637e+06
mean     6.062467e+01
std      4.345553e+01
min      5.000000e+00
25%      3.300000e+01
50%      5.000000e+01
75%      7.600000e+01
max      1.216000e+03
Name: review_text, dtype: float64

In [29]:
df.shape

(1090637, 9)

1,680 reviews under 5 words removed. Average word count remains ~60,
confirming the dataset is predominantly rich in content. Minimum word
count is now exactly 5 — filter confirmed working correctly.

**Remove Incentivized Reviews**

In [30]:
incentive_phrases = [
    "received this product for free",
    "incentivized review",
    "complimentary from",
    "gifted by"
]

pattern = '|'.join(incentive_phrases)

df = df[~df['review_text'].str.lower().str.contains(pattern, na=False)]

In [31]:
df.shape

(1033710, 9)

~ 56,927 incentivized reviews removed. This is the largest single
cleaning step (~5% of dataset). These reviews introduce structural
positive bias — reviewers who receive free products express more
positive sentiment regardless of actual product performance. Removing
them produces a cleaner, more representative sentiment signal.

**Text Normalization**

In [32]:
import re

In [33]:
df['clean_text'] = df['review_text'].str.lower() #lowercase

In [34]:
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r"[^\w\s']", '', x)) #punctuation

In [35]:
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'\s+', ' ', x).strip()) #extra spaces

In [36]:
df[['review_text', 'clean_text']].head()

,review_text,clean_text
0,I use this with the Nudestix “Citrus Clean Bal...,i use this with the nudestix citrus clean balm...
1,I bought this lip mask after reading the revie...,i bought this lip mask after reading the revie...
2,My review title says it all! I get so excited ...,my review title says it all i get so excited t...
3,I’ve always loved this formula for a long time...,ive always loved this formula for a long time ...
4,"If you have dry cracked lips, this is a must h...",if you have dry cracked lips this is a must ha...


In [37]:
df['clean_text'].str.contains('[A-Z]').sum()

np.int64(0)

In [38]:
df['clean_text'].str.contains(r'[^\w\s]').sum()

np.int64(0)

In [39]:
df['clean_text'].str.contains('  ').sum()

np.int64(0)

**Text Normalization Verified:**
- Uppercase characters remaining: 0 
- Special characters remaining: 0 
- Double spaces remaining: 0 

`clean_text` is stored alongside the original `review_text`.
The original is preserved for transformer-based models (Stage 7 VADER
uses original text to preserve punctuation signals). `clean_text` is
used for TF-IDF and CountVectorizer analysis where consistent
tokenization matters.

**Known limitation:** Apostrophe removal converts contractions
(e.g., "I've" → "ive"). This is acceptable for frequency-based NLP
but would need addressing for transformer models in Phase 2.

**Standardize Skin Type**

In [40]:
df['skin_type'] = df['skin_type'].str.lower().str.strip()

In [41]:
df['skin_type'].unique()

array(['dry', nan, 'combination', 'normal', 'oily'], dtype=object)

**Skin Type Standardization Complete:**
Four clean values: dry, combination, normal, oily — plus NaN.
No mixed casing or whitespace artifacts remain. NaN retained for
~10.5% of records — demographic labels should not be fabricated.

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1033710 entries, 0 to 1094410
Data columns (total 10 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   product_id    1033710 non-null  object        
 1   product_name  1033710 non-null  object        
 2   brand_name    1033710 non-null  object        
 3   price_usd     1033710 non-null  float64       
 4   rating        1033710 non-null  int64         
 5   review_text   1033710 non-null  object        
 6   skin_type     924696 non-null   object        
 7   review_date   1033710 non-null  datetime64[ns]
 8   word_count    1033710 non-null  int64         
 9   clean_text    1033710 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 86.8+ MB


In [43]:
df.isnull().sum()

product_id           0
product_name         0
brand_name           0
price_usd            0
rating               0
review_text          0
skin_type       109014
review_date          0
word_count           0
clean_text           0
dtype: int64

In [44]:
df.to_csv("../sephora/clean_reviews.csv", index=False)

## Summary — Data Cleaning

### Cleaning Pipeline

| Step | Action | Rows Removed | Remaining |
|---|---|---|---|
| Start | Working dataset | — | 1,094,411 |
| 1 | Remove null review_text | 1,444 | 1,092,967 |
| 2 | Remove ingestion duplicates | 650 | 1,092,317 |
| 3 | Remove short reviews (<5 words) | 1,680 | 1,090,637 |
| 4 | Remove incentivized reviews | ~56,927 | 1,033,710 |

**Total removed:** 60,701 rows (~ 5.5%)
**Retained:** 1,033,710 reviews (~ 94.5%)

### Why 5.5% Removal Is the Right Amount

- Too little removal (<1%) would suggest superficial cleaning
- Too much removal (>20%) would suggest aggressive data loss
- 5.5% reflects targeted, justified removal — each row removed
  has a documented business or technical reason

### Text Normalization

- Lowercased all text
- Removed punctuation (apostrophes preserved via updated regex)
- Normalized whitespace
- Created `clean_text` alongside original `review_text`

### Final Dataset Profile

| Metric | Value |
|---|---|
| Total reviews | 1,033,710 |
| Avg word count | ~60 words |
| Median word count | 50 words |
| Min word count | 5 words |
| Skin type coverage | ~89.5% |

**Output file:** `clean_reviews.csv`

**Next:** Stage — EDA will explore patterns and validate
assumptions before deep analysis.